## Relevante Keywords finden

Anhand manuell extrahierter Zeitungsseiten aus den Jahren 1814-1900 (jeweils erste Juliwoche), die Rezensionen bzw. sonstige Texte mit Literaturthematik enthalten, wird in diesem Notebook versucht, relevante Keywords zu finden, die in zahlreichen Texten mit Literaturthematik und möglichst wenig Texten zu anderen Themen auftreten.

Ziel: Für die automatische Extraktion eine Vorauswahl an Seiten anhand von Keywords auswählen, die dann an die LLMs übergeben werden können und eine sinnvolle Balance aus Precision/Recall gewährleisten.

In [ ]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/rezensionen_for_keyword_extraction.xlsx")


df.columns = df.iloc[0]
df = df.iloc[1:]

df


,ID,pagenumber,day,publication_date,review_text,"category (werbung, gelehrt, rezension, information, notiz)",note,contains_keywords
1,1,4,Sa,06.07.1816,"Handbuch der Sprachwissenschaft,mit besonderer...",werbung,NaN,NaN
2,2,3,So,07.07.1816,"Bei Heinr. Rommerskirchen, Buchhändler dahier,...",werbung,NaN,NaN
3,3,4,So,07.07.1816,nur so piele Exemplare werden auf Postpapter a...,werbung,continues 2,NaN
4,4,3,Do,02.07.1818,Unser Planet oder die Erde in mathematischer u...,werbung,NaN,NaN
5,5,3,Sa,04.07.1818,"DüMont Schauberg in Köln ist zu18e erlangen, P...",werbung,NaN,NaN
...,...,...,...,...,...,...,...,...
496,496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,497,NaN,NaN,NaN,NaN,NaN,NaN,NaN
498,498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
499,499,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df = df.dropna(
    how="all",
    subset=df.columns.difference(["ID"])
)
df

,ID,pagenumber,day,publication_date,review_text,"category (werbung, gelehrt, rezension, information, notiz)",note,contains_keywords
1,1,4,Sa,06.07.1816,"Handbuch der Sprachwissenschaft,mit besonderer...",werbung,NaN,NaN
2,2,3,So,07.07.1816,"Bei Heinr. Rommerskirchen, Buchhändler dahier,...",werbung,NaN,NaN
3,3,4,So,07.07.1816,nur so piele Exemplare werden auf Postpapter a...,werbung,continues 2,NaN
4,4,3,Do,02.07.1818,Unser Planet oder die Erde in mathematischer u...,werbung,NaN,NaN
5,5,3,Sa,04.07.1818,"DüMont Schauberg in Köln ist zu18e erlangen, P...",werbung,NaN,NaN
...,...,...,...,...,...,...,...,...
471,471,9,Mi,05.07.1899,[Victor Cherbuliez.] Daß der am 2. d. M. erfol...,"bericht, rezension",interessanter Fall: negative Bewertungen in To...,NaN
472,472,6,Do,06.07.1899,Erfahrungen und Ratschläge eines alten Arztes....,rezension (gebrauch),NaN,NaN
473,473,9,So,1900-07-01 00:00:00,Literatur.Rechtswissenschaft.* Vergleichende D...,rezension (gebrauch),NaN,NaN
474,474,10,So,1900-07-01 00:00:00,die Ausführungs=Verordnungen und=Verfügungen. ...,rezension (gebrauch),continues 473,NaN


### Funktion: Ist ein Keyword im übergebenen Text enthalten?

In [ ]:
def extract_keywords(text, keywords):
    if pd.isna(text):
        return []

    text_norm = text.lower()
    return [kw for kw in keywords if kw.lower() in text_norm]



In [ ]:
keywords = ["Kapitel", "Capitel", "Literatur", "Litteratur", "literarisch", "litterarisch","Gedicht", "Roman", "Neuerscheinung", "Dichter", "Schriftsteller", "Schriftst.", "Rezensent", "Recension", "Recensent", "erschienen", "erscheinen", "Werk", "Schrift", "Rezension", "Buch", "Autor", "Verfasser", "verfasst", "herausgegeben", "Herausgeber", "Uebersetzung", "Übersetzung", "übersetzt", "uebersetzt", "lesen", "lesenswert", "Leser", "Büchlein", "Buechlein", "Schrift", "Werk", "Poesie", "Poet"]



In [ ]:
df["contains_keywords"] = df["review_text"].apply(
    extract_keywords,
    keywords=keywords
)

In [ ]:
df.to_excel("july_df_with_keywords.xlsx", index=False)

In [ ]:
exploded = df.explode("contains_keywords")


In [ ]:
exploded = exploded.dropna(subset=["contains_keywords"])


In [ ]:
total_counts = (
    exploded
    .groupby("contains_keywords")
    .size()
    .rename("count_total")
)

rezension_mask = exploded["category (werbung, gelehrt, rezension, information, notiz)"].str.contains(
    "rezension",
    case=False,
    na=False
)

rezension_counts = (
    exploded[rezension_mask]
    .groupby("contains_keywords")
    .size()
    .rename("count_rezension")
)


In [ ]:
result = (
    pd.concat([total_counts, rezension_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"contains_keywords": "keyword"})
)


In [ ]:
print(result)

           keyword  count_total  count_rezension
0            Autor           31               25
1             Buch          332              150
2         Büchlein           33               19
3          Capitel           12                7
4          Dichter           76               56
5          Gedicht           66               37
6      Herausgeber           29               18
7          Kapitel            2                1
8            Leser           69               55
9        Literatur          119               94
10      Litteratur           14                4
11          Poesie           24               19
12            Poet           37               27
13       Recension            5                1
14       Rezension            1                0
15           Roman           80               55
16         Schrift          446              234
17  Schriftsteller           53               38
18    Uebersetzung           50               27
19       Verfasser  

In [ ]:
result.to_excel("keyword_counts.xlsx", index=False)

### Der nächste Schritt

Wir wollen ein Dataframe mit allen Artikeln aus den jeweiligen Jahren im entsprechenden Zeitraum erstellen.

In [ ]:
!pip install ddbapi
from ddbapi import zp_pages, list_column, filter

  Preparing metadata (setup.py) ... done
  Created wheel for ddbapi: filename=ddbapi-0.1.2-py3-none-any.whl size=5381 sha256=ad4eb6024f7d9918fceb18a0c7dcd5fa394dcd4e393a119444739e8ef6fcde24
  Stored in directory: /root/.cache/pip/wheels/b9/fa/90/7b6f9ccac9679cba8b348ae6ba003902e1fdc4d44d4b613c38
Successfully built ddbapi


In [ ]:
import pandas as pd

dfs = []

for year in range(1814, 1901):  # inklusive 1900
    date_range = f'[{year}-07-01T12:00:00Z TO {year}-07-07T12:00:00Z]'
    print(f"Fetching {year}...")

    df_year = zp_pages(
        publication_date=date_range,
        paper_title='Kölnische Zeitung'
        # kein plainpagefulltext-Filter, somit alle Seiten
    )

    dfs.append(df_year)

# alle Jahre zusammenführen
huge_df = pd.concat(dfs, ignore_index=True)


Fetching 1814...
https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select?rows=1000&sort=id+ASC&q=type%3Apage+AND+publication_date%3A%22%5B1814-07-01T12%3A00%3A00Z%5C+TO%5C+1814-07-07T12%3A00%3A00Z%5D%22+AND+paper_title%3A%22K%C3%B6lnische%5C+Zeitung%22&cursorMark=%2A
Got 16 items.
Fetching 1815...
https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select?rows=1000&sort=id+ASC&q=type%3Apage+AND+publication_date%3A%22%5B1815-07-01T12%3A00%3A00Z%5C+TO%5C+1815-07-07T12%3A00%3A00Z%5D%22+AND+paper_title%3A%22K%C3%B6lnische%5C+Zeitung%22&cursorMark=%2A
Got 26 items.
Fetching 1816...
https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select?rows=1000&sort=id+ASC&q=type%3Apage+AND+publication_date%3A%22%5B1816-07-01T12%3A00%3A00Z%5C+TO%5C+1816-07-07T12%3A00%3A00Z%5D%22+AND+paper_title%3A%22K%C3%B6lnische%5C+Zeitung%22&cursorMark=%2A
Got 20 items.
Fetching 1817...
https://api.deutsche-digitale-bibliothek.de/search/index/newsp

In [ ]:
huge_df["contains_keywords"] = huge_df["plainpagefulltext"].apply(
    extract_keywords,
    keywords=keywords
)


In [ ]:
huge_df.to_csv("july_df_with_keywords.csv")

In [ ]:
exploded = huge_df.explode("contains_keywords")
exploded = exploded.dropna(subset=["contains_keywords"])


In [ ]:
all_pages_counts = exploded.groupby("contains_keywords").size().rename("all_pages")


In [ ]:
result = result.merge(
    all_pages_counts.rename("all_pages"),
    how="left",
    left_on="keyword",
    right_index=True
)

# fehlende Keywords = 0
result["all_pages"] = result["all_pages"].fillna(0).astype(int)


In [ ]:
print(result)

           keyword  count_total  count_rezension  all_pages
0            Autor           31               25        697
1             Buch          332              150       3253
2         Büchlein           33               19         63
3          Capitel           12                7        235
4          Dichter           76               56        437
5          Gedicht           66               37        266
6      Herausgeber           29               18        186
7          Kapitel            2                1         22
8            Leser           69               55        539
9        Literatur          119               94        439
10      Litteratur           14                4         39
11          Poesie           24               19        103
12            Poet           37               27        172
13       Recension            5                1         15
14       Rezension            1                0          2
15           Roman           80         

In [ ]:
# 1️⃣ Spalten umbenennen
result = result.rename(columns={
    "count_total": "count_literature_themed",
    "count_rezension": "count_review"
})

# 2️⃣ Spaltenreihenfolge anpassen
result = result[["keyword", "all_pages", "count_literature_themed", "count_review"]]

# 3️⃣ Optional: anzeigen
print(result.head())


    keyword  all_pages  count_literature_themed  count_review
0     Autor        697                       31            25
1      Buch       3253                      332           150
2  Büchlein         63                       33            19
3   Capitel        235                       12             7
4   Dichter        437                       76            56


In [ ]:
# 1️⃣ all_pages umbenennen
result = result.rename(columns={"all_pages": "count_all_pages"})

# 2️⃣ Prozentwerte als Kommazahlen (0 bis 1)
result["relevant_literature_themed"] = result["count_literature_themed"] / result["count_all_pages"]
result["relevant_review"] = result["count_review"] / result["count_all_pages"]

# 3️⃣ Spaltenreihenfolge
result = result[[
    "keyword",
    "count_all_pages",
    "count_literature_themed",
    "count_review",
    "relevant_literature_themed",
    "relevant_review"
]]

# 4️⃣ Optional: anzeigen
print(result.head())


    keyword  count_all_pages  count_literature_themed  count_review  \
0     Autor              697                       31            25   
1      Buch             3253                      332           150   
2  Büchlein               63                       33            19   
3   Capitel              235                       12             7   
4   Dichter              437                       76            56   

   relevant_literature_themed  relevant_review  
0                    0.044476         0.035868  
1                    0.102060         0.046111  
2                    0.523810         0.301587  
3                    0.051064         0.029787  
4                    0.173913         0.128146  


In [ ]:
# Export als Excel-Datei
result.to_excel("keyword_summary.xlsx", index=False)


In [ ]:
# Filter für Keywords mit mehr als 10% in count_review
keywords_over_10pct = result[result["relevant_review"] > 0.1]["keyword"].tolist()

# Ausgabe
print(keywords_over_10pct)


['Büchlein', 'Dichter', 'Gedicht', 'Leser', 'Literatur', 'Litteratur', 'Poesie', 'Poet', 'Schriftsteller', 'Uebersetzung', 'Verfasser', 'herausgegeben', 'lesenswert', 'literarisch', 'uebersetzt', 'übersetzt']


In [ ]:
rezension_df = df[df["category (werbung, gelehrt, rezension, information, notiz)"].str.contains("rezension", case=False, na=False)].copy()


In [ ]:
def has_top_keyword(keyword_list):
    if not keyword_list:  # leere Liste
        return False
    return any(kw in keywords_over_10pct for kw in keyword_list)

rezension_df["has_top_keyword"] = rezension_df["contains_keywords"].apply(has_top_keyword)


In [ ]:
coverage = rezension_df["has_top_keyword"].mean()
print(f"Abdeckung der Rezensionen durch Top-Keywords: {coverage:.2%}")


Abdeckung der Rezensionen durch Top-Keywords: 90.30%


In [ ]:
top_review_keywords = result.sort_values(by="count_review", ascending=False)

print(top_review_keywords.head(10))

       keyword  count_all_pages  count_literature_themed  count_review  \
20        Werk             7152                      528           286   
16     Schrift             6546                      446           234   
1         Buch             3253                      332           150   
19   Verfasser              672                      194           133   
22  erschienen             1501                      263           107   
9    Literatur              439                      119            94   
4      Dichter              437                       76            56   
8        Leser              539                       69            55   
15       Roman              777                       80            55   
21  erscheinen             1546                      101            54   

    relevant_literature_themed  relevant_review  
20                    0.073826         0.039989  
16                    0.068133         0.035747  
1                     0.102060     

### Transfer für größeren Zeitraum: Alle Seiten der Kölnischen Zeitung zwischen 1814 bis 1900 extrahieren, danach nach den ausgewählten Keywords filtern.

In [ ]:
!pip install ddbapi
import pandas as pd
from ddbapi import zp_pages, list_column, filter

In [ ]:
relevant_keywords = ['Büchlein', 'Dichter', 'Gedicht', 'Leser', 'Literatur', 'Litteratur', 'Poesie', 'Poet', 'Schriftsteller', 'Uebersetzung', 'Verfasser', 'herausgegeben', 'lesenswert', 'literarisch', 'uebersetzt', 'übersetzt']


In [ ]:
df_to_be_filtered = zp_pages(
        publication_date='[1814-01-01T12:00:00Z TO 1900-12-31T12:00:00Z]',
        paper_title='Kölnische Zeitung'
        # kein plainpagefulltext-Filter, somit alle Seiten
        )

https://api.deutsche-digitale-bibliothek.de/search/index/newspaper-issues/select?rows=1000&sort=id+ASC&q=type%3Apage+AND+publication_date%3A%22%5B1814-01-01T12%3A00%3A00Z%5C+TO%5C+1900-12-31T12%3A00%3A00Z%5D%22+AND+paper_title%3A%22K%C3%B6lnische%5C+Zeitung%22&cursorMark=%2A
Getting 1000 of 224927
Getting 2000 of 224927
Getting 3000 of 224927
Getting 4000 of 224927
Getting 5000 of 224927
Getting 6000 of 224927
Getting 7000 of 224927
Getting 8000 of 224927
Getting 9000 of 224927
Getting 10000 of 224927
Getting 11000 of 224927
Getting 12000 of 224927
Getting 13000 of 224927
Getting 14000 of 224927
Getting 15000 of 224927
Getting 16000 of 224927
Getting 17000 of 224927
Getting 18000 of 224927
Getting 19000 of 224927
Getting 20000 of 224927
Getting 21000 of 224927
Getting 22000 of 224927
Getting 23000 of 224927
Getting 24000 of 224927
Getting 25000 of 224927
Getting 26000 of 224927
Getting 27000 of 224927
Getting 28000 of 224927
Getting 29000 of 224927
Getting 30000 of 224927
Getting 31000

In [ ]:
df_to_be_filtered["contains_keywords"] = df_to_be_filtered["plainpagefulltext"].apply(
    extract_keywords,
    keywords=relevant_keywords
)

In [ ]:
# Kopieren, in dem kopierten df alle Einträge droppen, die keine Keywords enthalten. Beide (keyword_df, df_to_be_filtered)als csv exportieren

In [ ]:
# 1️⃣ Gesamtes DataFrame exportieren
#df_to_be_filtered.to_csv("df_to_be_filtered_all.csv", index=False)

# 2️⃣ Nur Einträge exportieren, die mindestens ein Keyword enthalten
df_to_be_filtered[df_to_be_filtered["contains_keywords"].map(bool)] \
   .to_csv("df_full_keywords.csv", index=False)


In [ ]:
from google.colab import files
files.download("/content/df_full_keywords.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>